# Surrogate Fidelity — public paper figures

This notebook regenerates the revised paper's public-data figures and tables from a clean checkout. Scalar results use the canonical full-dialog, pairwise-complete analysis in `results/f_table.tsv`; multiclass ANLI and RACE results use finite-extreme replacement for partially censored label scores while preserving all-label-missing rows as unavailable.

The notebook intentionally excludes CKA and other analyses that require unreleased hidden states or analysis code. It writes paper-ready outputs below `paper_outputs/` by default. Set `SURROGATE_PAPER_OUTPUT_DIR` to a paper source directory to write there directly.


In [ ]:
import importlib
import importlib.util
import os
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import gaussian_kde

try:
    from IPython.display import display
except ImportError:
    def display(value: object) -> None:
        print(value)


# Repository and output locations. Jupyter kernels need not start inside
# the checkout, so also resolve an editable installation of `surrogate`.
def _find_repo_root() -> Path:
    candidates: list[Path] = []
    override = os.environ.get("SURROGATE_REPO_ROOT")
    if override:
        candidates.append(Path(override).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.extend((cwd, *cwd.parents))
    package_spec = importlib.util.find_spec("surrogate")
    if package_spec is not None and package_spec.submodule_search_locations:
        candidates.extend(
            Path(location).resolve().parent
            for location in package_spec.submodule_search_locations
        )

    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "results" / "f_table.tsv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the surrogate checkout. Install it with `pip install -e .`, "
        "start the kernel inside the checkout, or set SURROGATE_REPO_ROOT."
    )


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
RESULTS_DIR = REPO_ROOT / "results"
OUTPUT_ROOT = Path(
    os.environ.get("SURROGATE_PAPER_OUTPUT_DIR", REPO_ROOT / "paper_outputs")
).resolve()
FIGURES_DIR = OUTPUT_ROOT / "figures" / "0625_cameraready"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
WRITE_PNG_PREVIEWS = os.environ.get("SURROGATE_PAPER_PNG_PREVIEWS", "0") == "1"
ICLR_TEXT_WIDTH_IN = 5.5
ICLR_WRAP_WIDTH_IN = ICLR_TEXT_WIDTH_IN / 2.0


# One semantic palette for the paper.
PALETTE = {
    "F_pred": "#584486",       # violet: black-box prediction fidelity
    "F_attr": "#0091A1",       # teal: black-box attribution fidelity
    "F_repr": "#BA7C43",       # orange: generic representation fidelity
    "F_mag": "#8E5313",        # dark orange: representation magnitude
    "F_align": "#E9A86E",      # light orange: representation alignment
    "F_attn": "#8594A6",       # steel: attention family
    "F_cross": None,              # cross-level: source color plus marker/hatch
    "control": "#9A958F",      # warm gray: controls and nulls
    "neutral": "#2E2E38",      # ink: neutral derived quantities
    "class_false": "#2E2E38",  # ink
    "class_true": "#BA7C43",   # middle orange
}
ATTENTION_LINESTYLES = {
    "F_attn_mean": "-",
    "F_attn_max": "--",
    "F_attn_rollout": ":",
}
LENS_LINESTYLES = {
    "tuned": "-",
    "untuned": "--",
}

OPEN_MODELS = [
    "qwen2.5-0.5b-instruct",
    "qwen2.5-3b-instruct",
    "llama-3.1-8b-instruct",
    "qwen2.5-7b-instruct",
    "qwen2.5-14b-instruct",
]
HOSTED_MODELS = [
    "llama3.1-70b-instruct",
    "llama3.3-70b-instruct",
    "llama4-maverick-17b-128e-instruct",
    "gpt-4o",
    "gpt-4-1",
    "gemini-2-5-flash-lite-vertex",
]
MODELS = OPEN_MODELS + HOSTED_MODELS
MODEL_LABELS = {
    "qwen2.5-0.5b-instruct": "Qwen-0.5B",
    "qwen2.5-3b-instruct": "Qwen-3B",
    "llama-3.1-8b-instruct": "Llama-8B",
    "qwen2.5-7b-instruct": "Qwen-7B",
    "qwen2.5-14b-instruct": "Qwen-14B",
    "llama3.1-70b-instruct": "Llama-70B",
    "llama3.3-70b-instruct": "Llama-3.3-70B",
    "llama4-maverick-17b-128e-instruct": "Maverick",
    "gpt-4o": "GPT-4o",
    "gpt-4-1": "GPT-4.1",
    "gemini-2-5-flash-lite-vertex": "Gemini Flash",
}

# Start from upstream Matplotlib defaults rather than inheriting a Meta/Bento style.
plt.rcdefaults()
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Nimbus Roman", "Times New Roman", "Times", "STIXGeneral"],
    "mathtext.fontset": "stix",
    "font.size": 8.0,
    "axes.titlesize": 9.0,
    "axes.labelsize": 8.0,
    "legend.fontsize": 7,
    "xtick.labelsize": 7.0,
    "ytick.labelsize": 7.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def _oklch_to_srgb(lightness: float, chroma: float, hue: float) -> tuple[float, float, float]:
    """Convert one OKLCH color to clipped display sRGB."""
    angle = np.deg2rad(hue)
    a = chroma * np.cos(angle)
    b = chroma * np.sin(angle)
    l_ = lightness + 0.3963377774 * a + 0.2158037573 * b
    m_ = lightness - 0.1055613458 * a - 0.0638541728 * b
    s_ = lightness - 0.0894841775 * a - 1.2914855480 * b
    l, m, s = l_**3, m_**3, s_**3
    linear = np.array([
        4.0767416621 * l - 3.3077115913 * m + 0.2309699292 * s,
        -1.2684380046 * l + 2.6097574011 * m - 0.3413193965 * s,
        -0.0041960863 * l - 0.7034186147 * m + 1.7076147010 * s,
    ])
    rgb = np.where(
        linear <= 0.0031308,
        12.92 * linear,
        1.055 * np.maximum(linear, 0.0) ** (1.0 / 2.4) - 0.055,
    )
    return tuple(np.clip(rgb, 0.0, 1.0))


def _gamut_mapped_oklch(lightness: float, chroma: float, hue: float) -> tuple[float, float, float]:
    """Reduce chroma at fixed lightness and hue until the color is in gamut."""
    low, high = 0.0, chroma
    for _ in range(24):
        candidate = (low + high) / 2.0
        angle = np.deg2rad(hue)
        a, b = candidate * np.cos(angle), candidate * np.sin(angle)
        l_ = lightness + 0.3963377774 * a + 0.2158037573 * b
        m_ = lightness - 0.1055613458 * a - 0.0638541728 * b
        s_ = lightness - 0.0894841775 * a - 1.2914855480 * b
        l, m, s = l_**3, m_**3, s_**3
        linear = np.array([
            4.0767416621 * l - 3.3077115913 * m + 0.2309699292 * s,
            -1.2684380046 * l + 2.6097574011 * m - 0.3413193965 * s,
            -0.0041960863 * l - 0.7034186147 * m + 1.7076147010 * s,
        ])
        if np.all((linear >= 0.0) & (linear <= 1.0)):
            low = candidate
        else:
            high = candidate
    return _oklch_to_srgb(lightness, low, hue)


def fidelity_ramp(name: str, hue: float, chroma_scale: float = 1.0) -> mcolors.ListedColormap:
    """Create a fixed-lightness-profile OKLCH ramp for an r² heatmap."""
    positions = np.linspace(0.0, 1.0, 256)
    lightness = 0.97 - 0.77 * positions
    chroma = 0.106 * chroma_scale * np.sin(np.pi * positions) ** 0.75
    colors = [
        _gamut_mapped_oklch(float(L), float(C), hue)
        for L, C in zip(lightness, chroma)
    ]
    return mcolors.ListedColormap(colors, name=name)


CMAP_PRED = fidelity_ramp("f_pred_oklch", 295.0)
CMAP_ATTR = fidelity_ramp("f_attr_oklch", 208.0)
CMAP_MAG = mcolors.LinearSegmentedColormap.from_list(
    "f_mag_tonal", ["#FBF8F3", PALETTE["F_mag"]]
)
CMAP_ALIGN = mcolors.LinearSegmentedColormap.from_list(
    "f_align_tonal", ["#FFF9F4", PALETTE["F_align"]]
)
CMAP_REPR = mcolors.LinearSegmentedColormap.from_list(
    "f_repr_tonal", ["#FBF8F3", PALETTE["F_repr"]]
)
CMAP_CROSS = mcolors.LinearSegmentedColormap.from_list(
    "f_cross_neutral", ["#F7F6F4", PALETTE["neutral"]]
)


def save_figure(
    figure: plt.Figure, filename: str, *, exact_canvas: bool = False
) -> Path:
    """Save one stable paper PDF, optionally preserving its exact canvas."""
    path = FIGURES_DIR / filename
    crop_options = {} if exact_canvas else {"bbox_inches": "tight", "pad_inches": 0.04}
    figure.savefig(
        path,
        metadata={
            "Creator": "notebooks/paper_figures.ipynb",
            "CreationDate": None,
            "ModDate": None,
        },
        **crop_options,
    )
    if WRITE_PNG_PREVIEWS:
        figure.savefig(path.with_suffix(".png"), dpi=180, **crop_options)
    plt.close(figure)
    print(path.relative_to(OUTPUT_ROOT))
    return path


def format_unit_decimal(value: float, precision: int) -> str:
    """Format a bounded decimal without a leading zero."""
    formatted = f"{value:.{precision}f}"
    if formatted.startswith("0."):
        return formatted[1:]
    if formatted.startswith("-0."):
        return "-" + formatted[2:]
    return formatted


print(f"Repository: {REPO_ROOT}")
print(f"Outputs:    {OUTPUT_ROOT}")


## Load and validate the released tables

BoolQ and WinoGrande use pairwise-complete Pearson $r^2$. ANLI and RACE prediction and attribution use centered RV over all pairwise label margins, with model-specific finite floors for partial top-$k$ censoring and all-label-missing rows left unavailable. Scalar ANLI representation and cross-fidelity rows retain the entailment-minus-contradiction readout.


In [ ]:
F_TABLE = pd.read_csv(RESULTS_DIR / "f_table.tsv", sep="\t")
RACE_RV = pd.read_csv(RESULTS_DIR / "race_rv.tsv", sep="\t")
ANLI_RV = pd.read_csv(RESULTS_DIR / "anli_rv.tsv", sep="\t")
MULTICLASS_FLOOR = pd.read_csv(RESULTS_DIR / "multiclass_floor_sensitivity.tsv", sep="\t")
LAYER_CONTROL = pd.read_csv(
    REPO_ROOT / "layer_controls" / "layer_control_fidelity.tsv", sep="\t"
)
TUNED_LENS = pd.read_csv(
    REPO_ROOT / "layer_controls" / "tuned_lens_boolq_fidelity.tsv", sep="\t"
)

assert set(F_TABLE["scope"]) == {"all"}
assert set(F_TABLE["api_infinity_policy"]) == {"pairwise_complete"}
assert set(F_TABLE.loc[F_TABLE["benchmark"].str.startswith("anli"), "contrast"]) == {
    "entailment_contradiction"
}
assert set(RACE_RV["scope"]) == {"all"}
assert set(RACE_RV["missingness_policy"]) == {"model_specific_finite_extreme_preserve_all_missing"}
assert set(ANLI_RV["scope"]) == {"all"}
assert set(ANLI_RV["missingness_policy"]) == {"model_specific_finite_extreme_preserve_all_missing"}
assert set(MULTICLASS_FLOOR["scope"]) == {"all"}
assert set(LAYER_CONTROL["scope"]) == {"user"}
assert set(TUNED_LENS["analysis_status"]) == {"exploratory_sensitivity"}
assert set(TUNED_LENS["scope"]) == {"user"}


def canonical_rows(
    *,
    benchmark: str,
    pregrouper: str,
    metric: str | None = None,
    statistic: str = "pearson_r2",
) -> pd.DataFrame:
    contrast = "entailment_contradiction" if benchmark.startswith("anli") else "canonical"
    mask = (
        F_TABLE["benchmark"].eq(benchmark)
        & F_TABLE["pregrouper"].eq(pregrouper)
        & F_TABLE["scope"].eq("all")
        & F_TABLE["contrast"].eq(contrast)
        & F_TABLE["statistic"].eq(statistic)
        & F_TABLE["availability_status"].eq("available")
        & F_TABLE["api_infinity_policy"].eq("pairwise_complete")
    )
    if metric is not None:
        mask &= F_TABLE["metric"].eq(metric)
    return F_TABLE.loc[mask].copy()


BOOLQ_R2 = canonical_rows(benchmark="boolq", pregrouper="sentence")
assert len(BOOLQ_R2.loc[BOOLQ_R2["metric"].eq("F_pred")]) == 55
assert len(BOOLQ_R2.loc[BOOLQ_R2["metric"].eq("F_attr")]) == 55
print("Canonical BoolQ pair rows:", len(BOOLQ_R2))


## Main summary table

In [ ]:
METRIC_LABELS = {
    "F_pred": r"$F_{\mathrm{pred}}$",
    "F_attr": r"$F_{\mathrm{attr}}$",
    "F_attn_mean": r"$F_{\mathrm{attn}}^{\mathrm{mean}}$",
    "F_attn_max": r"$F_{\mathrm{attn}}^{\mathrm{max}}$",
    "F_attn_rollout": r"$F_{\mathrm{attn}}^{\mathrm{rollout}}$",
    "F_mag": r"$F_{\mathrm{mag}}$",
    "F_align": r"$F_{\mathrm{align}}$",
    "F_mag_to_attr": r"$F_{\mathrm{mag}\to|\mathrm{attr}|}$",
    "F_align_to_attr": r"$F_{\mathrm{align}\to\mathrm{attr}}$",
    "F_attn_mean_to_attr": r"$F_{\mathrm{attn}\to\mathrm{attr}}^{\mathrm{mean}}$",
    "F_attn_max_to_attr": r"$F_{\mathrm{attn}\to\mathrm{attr}}^{\mathrm{max}}$",
    "F_attn_rollout_to_attr": r"$F_{\mathrm{attn}\to\mathrm{attr}}^{\mathrm{rollout}}$",
}

SUMMARY_GROUPS = [
    ("Black-box", ["F_pred", "F_attr"]),
    ("Representation-level", [
        "F_attn_mean", "F_attn_max", "F_attn_rollout", "F_mag", "F_align",
    ]),
    ("Mechanistic-to-causal", [
        "F_mag_to_attr", "F_align_to_attr", "F_attn_mean_to_attr",
        "F_attn_max_to_attr", "F_attn_rollout_to_attr",
    ]),
]


def _triplet(values: pd.Series) -> str:
    finite = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if not len(finite):
        return "---"
    low, median, high = np.min(finite), np.median(finite), np.max(finite)
    return " / ".join(format_unit_decimal(value, 3) for value in (low, median, high))


summary_records = []
for group, metrics in SUMMARY_GROUPS:
    for metric in metrics:
        rows = BOOLQ_R2.loc[BOOLQ_R2["metric"].eq(metric)]
        symmetric_black_box = metric in {"F_pred", "F_attr"}
        representation_only = metric in {
            "F_attn_mean", "F_attn_max", "F_attn_rollout", "F_mag", "F_align"
        }
        broad_pops = (
            {"open_open", "open_hosted", "hosted_hosted"}
            if symmetric_black_box
            else {"open_open", "open_hosted"}
        )
        summary_records.append({
            "Group": group,
            "Metric": METRIC_LABELS[metric],
            "Open→Open": _triplet(rows.loc[rows["pair_population"].eq("open_open"), "f_point"]),
            "Open→Hosted": (
                "---" if representation_only else
                _triplet(rows.loc[rows["pair_population"].eq("open_hosted"), "f_point"])
            ),
            "All↔All / Open→All": (
                "---" if representation_only else
                _triplet(rows.loc[rows["pair_population"].isin(broad_pops), "f_point"])
            ),
        })

SUMMARY_TABLE = pd.DataFrame(summary_records)
display(SUMMARY_TABLE)

latex_lines = [
    r"% Canonical full-dialog, pairwise-complete Pearson r^2; entries are min / median / max.",
    r"% The final column is All<->All for black-box rows and Open->All for cross-level rows.",
    r"\begin{tabular}{llccc}",
    r"\toprule",
    r"Group & Metric & Open$\to$Open & Open$\to$Hosted & All$\leftrightarrow$All / Open$\to$All \\",
    r"\midrule",
]
last_group = None
for record in summary_records:
    if last_group is not None and record["Group"] != last_group:
        latex_lines.append(r"\midrule")
    latex_lines.append(
        f"{record['Group'] if record['Group'] != last_group else ''} & {record['Metric']} & "
        f"{record['Open→Open']} & {record['Open→Hosted']} & "
        f"{record['All↔All / Open→All']} \\\\" 
    )
    last_group = record["Group"]
latex_lines.extend([r"\bottomrule", r"\end{tabular}"])
(TABLES_DIR / "summary_pearson.tex").write_text("\n".join(latex_lines) + "\n")
print(TABLES_DIR / "summary_pearson.tex")


## Figure 3a — BoolQ fidelity dashboard

Both triangles use the same fixed $[0,1]$ scale and identical OKLCH lightness profiles. The upper triangle is $F_{\mathrm{pred}}$; the lower is signed $F_{\mathrm{attr}}$.


In [ ]:
def pair_grid(frame: pd.DataFrame, models: list[str], metric: str) -> np.ndarray:
    metric_rows = frame.loc[frame["metric"].eq(metric)]
    grid = np.full((len(models), len(models)), np.nan, dtype=float)
    model_index = {model: index for index, model in enumerate(models)}
    for row in metric_rows.itertuples(index=False):
        first, second = model_index[row.model_s], model_index[row.model_t]
        grid[first, second] = grid[second, first] = float(row.f_point)
    return grid


def plot_fidelity_dashboard(
    frame: pd.DataFrame,
    models: list[str],
    heatmap_filename: str,
    *,
    height: float,
) -> None:
    pred_grid = pair_grid(frame, models, "F_pred")
    attr_grid = pair_grid(frame, models, "F_attr")
    indices = np.indices(pred_grid.shape)
    upper_pred = np.where(indices[0] < indices[1], pred_grid, np.nan)
    lower_attr = np.where(indices[0] > indices[1], attr_grid, np.nan)
    annotation_size = 5.8 if len(models) == len(MODELS) else 5.3

    fig, ax = plt.subplots(
        figsize=(ICLR_WRAP_WIDTH_IN, height), layout="constrained"
    )
    ax.imshow(upper_pred, cmap=CMAP_PRED, vmin=0.0, vmax=1.0)
    ax.imshow(lower_attr, cmap=CMAP_ATTR, vmin=0.0, vmax=1.0)
    for row in range(len(models)):
        for column in range(len(models)):
            if row == column:
                continue
            value = pred_grid[row, column] if row < column else attr_grid[row, column]
            if np.isfinite(value):
                ax.text(
                    column, row, format_unit_decimal(value, 3), ha="center", va="center",
                    fontsize=annotation_size,
                    color="white" if value > 0.68 else "#17171C",
                )
    ax.set_xticks(
        range(len(models)), [MODEL_LABELS[m] for m in models], rotation=48, ha="right"
    )
    ax.set_yticks(range(len(models)), [MODEL_LABELS[m] for m in models])
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    save_figure(fig, heatmap_filename, exact_canvas=True)


plot_fidelity_dashboard(
    BOOLQ_R2, MODELS, "fig3a_fidelity_heatmap.pdf", height=2.70,
)


## Figure 3b+c — BoolQ cross-family contour panels

The open model is the locally evaluated Llama-3.1-8B, not the retained API serving-path diagnostic. Token aliases are aggregated with logsumexp before forming true-minus-false log-odds. Attribution uses original minus ablated log-odds over shared full-dialog coordinates.


In [ ]:
BOOLQ_SEGMENTS = pd.read_csv(RESULTS_DIR / "boolq" / "sentence" / "segments.tsv.gz", sep="\t")
ALL_KEYS = BOOLQ_SEGMENTS[["prompt_idx", "seg_idx"]]
assert len(ALL_KEYS) == 27_516


def _label_logsumexp(values: pd.Series) -> float:
    available = values.dropna().to_numpy(dtype=float)
    return float(logsumexp(available)) if len(available) else float("nan")


def load_boolq_signals(model: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    tokens = pd.read_csv(
        RESULTS_DIR / "boolq" / "sentence" / f"{model}_tokens.tsv.gz",
        sep="\t",
        dtype={"kind": "string", "answer": "string", "label": "string", "token": "string"},
    )
    tokens["label"] = tokens["label"].str.lower()
    tokens["seg_key"] = tokens["seg_idx"].fillna(-1).astype(int)
    grouped = (
        tokens.groupby(
            ["prompt_idx", "seg_key", "kind", "answer", "label"],
            sort=False, dropna=False, observed=True,
        )["logprob"]
        .agg(_label_logsumexp)
        .unstack("label")
        .reset_index()
    )
    grouped["logodds"] = grouped["true"] - grouped["false"]
    original = grouped.loc[
        grouped["kind"].eq("orig") & grouped["seg_key"].eq(-1),
        ["prompt_idx", "answer", "logodds"],
    ].drop_duplicates("prompt_idx")
    ablated = grouped.loc[
        grouped["kind"].eq("ablated"), ["prompt_idx", "seg_key", "logodds"]
    ].rename(columns={"seg_key": "seg_idx", "logodds": "ablated_logodds"})
    attribution = (
        ablated.merge(
            original.rename(columns={"logodds": "original_logodds"}),
            on="prompt_idx", how="inner", validate="many_to_one",
        )
        .merge(ALL_KEYS, on=["prompt_idx", "seg_idx"], how="inner", validate="one_to_one")
    )
    attribution["attribution"] = (
        attribution["original_logodds"] - attribution["ablated_logodds"]
    )
    return original, attribution


def paired_boolq_signals(source_model: str, target_model: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    source_prediction, source_attribution = load_boolq_signals(source_model)
    target_prediction, target_attribution = load_boolq_signals(target_model)
    prediction = source_prediction.merge(
        target_prediction[["prompt_idx", "logodds"]],
        on="prompt_idx", suffixes=("_source", "_target"), validate="one_to_one",
    )
    attribution = source_attribution.merge(
        target_attribution[["prompt_idx", "seg_idx", "attribution"]],
        on=["prompt_idx", "seg_idx"], suffixes=("_source", "_target"), validate="one_to_one",
    )
    prediction = prediction.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["logodds_source", "logodds_target"]
    )
    attribution = attribution.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["attribution_source", "attribution_target"]
    )
    return prediction, attribution


def _r_squared(frame: pd.DataFrame, x: str, y: str) -> float:
    correlation = float(frame[x].corr(frame[y]))
    return correlation * correlation


def _expanded_limits(
    values: pd.Series, lower_extra: float = 0.0, upper_extra: float = 0.0
) -> tuple[float, float]:
    lower, upper = np.quantile(values, [0.0005, 0.9995])
    span = max(upper - lower, np.finfo(float).eps)
    return (
        float(lower - span * (0.12 + lower_extra)),
        float(upper + span * (0.12 + upper_extra)),
    )


def _contour_panel(
    frame: pd.DataFrame,
    x: str,
    y: str,
    x_label: str,
    y_label: str,
    filename: str,
    extra_padding: tuple[float, float, float, float] = (0.0, 0.0, 0.0, 0.0),
    show_legend: bool = True,
) -> tuple[int, float]:
    x_minus, x_plus, y_minus, y_plus = extra_padding
    x_limits = _expanded_limits(frame[x], x_minus, x_plus)
    y_limits = _expanded_limits(frame[y], y_minus, y_plus)
    fig, ax = plt.subplots(
        figsize=(ICLR_WRAP_WIDTH_IN, 1.35), layout="constrained"
    )
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    for truth, color, label in [
        (False, PALETTE["class_false"], "False"),
        (True, PALETTE["class_true"], "True"),
    ]:
        selected = frame.loc[frame["answer"].str.lower().eq(str(truth).lower())]
        if len(selected) > 6_000:
            selected = selected.sample(6_000, random_state=42)
        values = selected[[x, y]].to_numpy(dtype=float).T
        density = gaussian_kde(values)
        xx, yy = np.meshgrid(
            np.linspace(*x_limits, 72), np.linspace(*y_limits, 72)
        )
        zz = density(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        levels = np.quantile(zz[zz > 0], [0.55, 0.72, 0.84, 0.92, 0.97])
        ax.contour(xx, yy, zz, levels=np.unique(levels), colors=color, linewidths=1.0, alpha=0.78)
        ax.plot([], [], color=color, label=label)
    r2 = _r_squared(frame, x, y)
    ax.text(0.04, 0.96, f"$r^2={r2:.3f}$", transform=ax.transAxes,
            ha="left", va="top")
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    if show_legend:
        ax.legend(
            frameon=False, loc="lower right", bbox_to_anchor=(1.01, -0.04),
            borderaxespad=0.0,
        )
    save_figure(fig, filename, exact_canvas=True)
    return len(frame), r2


PREDICTION_PAIR, ATTRIBUTION_PAIR = paired_boolq_signals(
    "llama-3.1-8b-instruct", "gpt-4o"
)
prediction_check = _contour_panel(
    PREDICTION_PAIR, "logodds_source", "logodds_target",
    "Llama-3.1-8B log-odds", "GPT-4o log-odds",
    "fig3b_logodds_contour.pdf",
    extra_padding=(0.0, 0.0, 0.0, 0.10),
    show_legend=False,
)
attribution_check = _contour_panel(
    ATTRIBUTION_PAIR, "attribution_source", "attribution_target",
    "Llama-3.1-8B attribution", "GPT-4o attribution",
    "fig3c_attribution_contour.pdf",
    extra_padding=(0.10, 0.10, 0.10, 0.10),
)
assert prediction_check[0] == 3_096 and np.isclose(prediction_check[1], 0.656005, atol=5e-6)
assert attribution_check[0] == 24_479 and np.isclose(attribution_check[1], 0.360039, atol=5e-6)


## Figure 4 — Attribution decomposition

These panels compare the public Llama-3.1-8B and Qwen-2.5-14B representation scalars over the full dialog. Alignment is the signed projection cosine $w\cdot\Delta z/(‖w‖‖\Delta z‖)$; it is not the cosine between original and perturbed representations. Dark, light, and middle orange distinguish perturbation magnitude, alignment, and the RMSNorm contribution, respectively. The magnitude-to-attribution row uses absolute attribution magnitude, as indicated by $F_{\mathrm{mag}\to|\mathrm{attr}|}$.


In [ ]:
def load_open_segments(model: str) -> pd.DataFrame:
    frame = pd.read_csv(
        RESULTS_DIR / "boolq" / "sentence" / f"{model}_segment.tsv.gz", sep="\t"
    )
    return frame.copy()


llama_segments = load_open_segments("llama-3.1-8b-instruct")
qwen_segments = load_open_segments("qwen2.5-14b-instruct")
REPRESENTATION_PAIR = llama_segments.merge(
    qwen_segments,
    on=["prompt_idx", "seg_idx"],
    suffixes=("_llama", "_qwen"),
    validate="one_to_one",
)

for suffix, width in [("llama", 4096), ("qwen", 5120)]:
    REPRESENTATION_PAIR[f"alignment_{suffix}"] = (
        REPRESENTATION_PAIR[f"w_dot_delta_z_postnorm_{suffix}"]
        / (
            REPRESENTATION_PAIR[f"w_norm_{suffix}"]
            * REPRESENTATION_PAIR[f"delta_norm_postnorm_{suffix}"]
        )
    )
    REPRESENTATION_PAIR[f"norm_contribution_{suffix}"] = (
        np.sqrt(width)
        * REPRESENTATION_PAIR[f"w_dot_z_pert_prenorm_{suffix}"]
        * (
            REPRESENTATION_PAIR[f"z_pert_norm_prenorm_{suffix}"]
            - REPRESENTATION_PAIR[f"z_orig_norm_prenorm_{suffix}"]
        )
        / (
            REPRESENTATION_PAIR[f"z_orig_norm_prenorm_{suffix}"]
            * REPRESENTATION_PAIR[f"z_pert_norm_prenorm_{suffix}"]
        )
    )


def _joint_distribution(
    frame: pd.DataFrame,
    x: str,
    y: str,
    label: str,
    color: str,
    cmap: mcolors.Colormap,
    filename: str,
    extent_padding: tuple[float, float, float, float],
    nonnegative: bool = False,
) -> float:
    finite = frame[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    x_lower, x_upper = np.quantile(finite[x], [0.005, 0.995])
    y_lower, y_upper = np.quantile(finite[y], [0.005, 0.995])
    x_span, y_span = x_upper - x_lower, y_upper - y_lower
    x_minus, x_plus, y_minus, y_plus = extent_padding
    x_limits = (x_lower - x_minus * x_span, x_upper + x_plus * x_span)
    y_limits = (y_lower - y_minus * y_span, y_upper + y_plus * y_span)
    if nonnegative:
        x_limits = (max(0.0, x_limits[0]), x_limits[1])
        y_limits = (max(0.0, y_limits[0]), y_limits[1])
    figure = plt.figure(figsize=(2.55, 2.55))
    grid = figure.add_gridspec(2, 2, width_ratios=(4, 1), height_ratios=(1, 4), hspace=0.05, wspace=0.05)
    top = figure.add_subplot(grid[0, 0])
    joint = figure.add_subplot(grid[1, 0], sharex=top)
    right = figure.add_subplot(grid[1, 1], sharey=joint)
    density_sample = finite if len(finite) <= 6_000 else finite.sample(6_000, random_state=42)
    density = gaussian_kde(density_sample[[x, y]].to_numpy(dtype=float).T)
    xx, yy = np.meshgrid(
        np.linspace(*x_limits, 80), np.linspace(*y_limits, 80)
    )
    zz = density(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    contour_levels = np.unique(
        np.r_[np.quantile(zz[zz > 0], [0.55, 0.72, 0.84, 0.92, 0.97]), zz.max()]
    )
    joint.contourf(xx, yy, zz, levels=contour_levels, cmap=cmap, alpha=0.82)
    joint.contour(
        xx, yy, zz, levels=contour_levels[:-1], colors=color, linewidths=0.55
    )
    x_values = np.linspace(*x_limits, 256)
    y_values = np.linspace(*y_limits, 256)
    x_density = gaussian_kde(density_sample[x].to_numpy(dtype=float))(x_values)
    y_density = gaussian_kde(density_sample[y].to_numpy(dtype=float))(y_values)
    top.fill_between(x_values, 0.0, x_density, color=color, alpha=0.14, linewidth=0.0)
    top.plot(x_values, x_density, color=color, linewidth=1.0)
    right.fill_betweenx(y_values, 0.0, y_density, color=color, alpha=0.14, linewidth=0.0)
    right.plot(y_density, y_values, color=color, linewidth=1.0, linestyle="--")
    r2 = _r_squared(finite, x, y)
    joint.text(0.04, 0.96, f"$r^2={r2:.3f}$", transform=joint.transAxes, ha="left", va="top")
    joint.set_xlabel(f"Llama-3.1-8B {label}")
    joint.set_ylabel(f"Qwen-2.5-14B {label}")
    top.axis("off")
    right.axis("off")
    save_figure(figure, filename)
    return r2


decomposition_checks = {
    "magnitude": _joint_distribution(
        REPRESENTATION_PAIR, "delta_norm_postnorm_llama", "delta_norm_postnorm_qwen",
        r"$\|\Delta z\|$", PALETTE["F_mag"], CMAP_MAG,
        "fig4a_delta_norm_contour.pdf",
        extent_padding=(0.12, 0.0, 0.12, 0.12), nonnegative=True,
    ),
    "alignment": _joint_distribution(
        REPRESENTATION_PAIR, "alignment_llama", "alignment_qwen",
        r"$\cos(\Delta z,w)$", PALETTE["F_align"], CMAP_ALIGN,
        "fig4b_alignment_contour.pdf",
        extent_padding=(0.0, 0.12, 0.0, 0.0),
    ),
    "normalization": _joint_distribution(
        REPRESENTATION_PAIR, "norm_contribution_llama", "norm_contribution_qwen",
        "RMSNorm contribution", PALETTE["F_repr"], CMAP_REPR,
        "fig4c_norm_contribution_contour.pdf",
        extent_padding=(0.22, 0.22, 0.22, 0.22),
    ),
}
expected_decomposition = {"magnitude": 0.855181, "alignment": 0.201733, "normalization": 0.064599}
for name, expected in expected_decomposition.items():
    assert np.isclose(decomposition_checks[name], expected, atol=5e-6), (name, decomposition_checks[name])
print(decomposition_checks)


## Figure 5 — Per-layer fidelity

This retained GPU sensitivity is restricted to user-message coordinates; unlike the canonical full-dialog tables and figures above, its stored random-control projections do not include system coordinates. The primary curves are actual, unnormalized Pearson $r^2$ values over ten open-model pairs. Models are aligned on a common relative-depth grid by linearly interpolating signed scores over decoder-block outputs; the embedding slot is excluded. Target ribbons are pointwise prompt-cluster bootstrap confidence intervals. The control band is the empirical 2.5–97.5% interval over grouped 9-vs-8 readout-compatible directions.

The grouped 9-vs-8 direction is the most defensible single control because it matches the grouped-logsumexp construction of headline $F_{\mathrm{attr}}$. Other null families remain available in the released layer-control table for appendix analyses.


In [ ]:
from benchmark_scripts import plot_layer_controls

# Bento kernels may retain an older imported plotting module across pulls.
plot_layer_controls = importlib.reload(plot_layer_controls)

layer_rows = LAYER_CONTROL.loc[LAYER_CONTROL["summary_kind"].eq("relative_depth")]
endpoint = layer_rows.sort_values("relative_depth_index").iloc[-1]
layer_endpoint = {
    "F_pred": endpoint["grouped_logsumexp_prediction_mean_pair_pearson_r2"],
    "F_attr": endpoint["grouped_logsumexp_attribution_mean_pair_pearson_r2"],
    "control": endpoint["grouped_9v8_pseudo_label_median"],
}
print(layer_endpoint)
layer_figure = FIGURES_DIR / "fig5_per_layer_fidelity.pdf"
plot_layer_controls.plot_summary(
    str(REPO_ROOT / "layer_controls" / "layer_control_fidelity.tsv"),
    str(layer_figure),
)


## Figure 6 — All-vs-all fidelity measurements

This expands the original cross-measurement matrix to all five open models. Every cell is pooled full-dialog Pearson $r^2$ on pairwise-complete observations. The six families are ablation, three attention aggregations, perturbation magnitude, and signed readout alignment; the fixed $[0,1]$ scale is shared with the other fidelity heatmaps.


In [ ]:
CROSS_SIGNAL_SPECS = [
    ("Ablation", "attribution", "F_attr"),
    ("Mean attn.", "attention_mean", "F_attn_mean"),
    ("Max attn.", "attention_max", "F_attn_max"),
    ("Rollout attn.", "attention_rollout", "F_attn_rollout"),
    (r"$\|\Delta z\|$", "delta_norm_postnorm", "F_mag"),
    (r"$\cos(\Delta z,w)$", "alignment", "F_align"),
]
OPEN_MODEL_SHORT_LABELS = {
    "qwen2.5-0.5b-instruct": "Q-0.5B",
    "qwen2.5-3b-instruct": "Q-3B",
    "llama-3.1-8b-instruct": "L-8B",
    "qwen2.5-7b-instruct": "Q-7B",
    "qwen2.5-14b-instruct": "Q-14B",
}


def _cross_signal_frame(model: str) -> pd.DataFrame:
    segments = load_open_segments(model).set_index(["prompt_idx", "seg_idx"])
    _, attribution = load_boolq_signals(model)
    attribution = attribution.set_index(["prompt_idx", "seg_idx"])[["attribution"]]
    frame = segments.join(attribution, how="inner", validate="one_to_one")
    frame["alignment"] = (
        frame["w_dot_delta_z_postnorm"]
        / (frame["w_norm"] * frame["delta_norm_postnorm"])
    )
    assert len(frame) == len(ALL_KEYS)
    return frame


cross_signal_frames = {model: _cross_signal_frame(model) for model in OPEN_MODELS}
cross_series = []
cross_labels = []
for family, column, _ in CROSS_SIGNAL_SPECS:
    for model in OPEN_MODELS:
        cross_series.append(cross_signal_frames[model][column].rename((family, model)))
        cross_labels.append(OPEN_MODEL_SHORT_LABELS[model])
cross_values = pd.concat(cross_series, axis=1).replace([np.inf, -np.inf], np.nan)
cross_r2 = cross_values.corr(min_periods=3).pow(2).to_numpy(dtype=float)
assert cross_r2.shape == (len(CROSS_SIGNAL_SPECS) * len(OPEN_MODELS),) * 2
assert np.allclose(np.diag(cross_r2), 1.0)

# Same-family blocks must reproduce the canonical released table exactly.
for family_index, (_, _, metric) in enumerate(CROSS_SIGNAL_SPECS):
    rows = BOOLQ_R2.loc[BOOLQ_R2["metric"].eq(metric)]
    for source_index, source in enumerate(OPEN_MODELS):
        for target_index, target in enumerate(OPEN_MODELS[source_index + 1:], source_index + 1):
            expected = rows.loc[
                rows["model_s"].eq(source) & rows["model_t"].eq(target), "f_point"
            ]
            assert len(expected) == 1
            row = family_index * len(OPEN_MODELS) + source_index
            column = family_index * len(OPEN_MODELS) + target_index
            assert np.isclose(cross_r2[row, column], float(expected.iloc[0]), atol=5e-12)

n_cross_signals = len(cross_labels)
indices = np.indices((n_cross_signals, n_cross_signals))
lower_triangle = np.where(indices[0] >= indices[1], cross_r2, np.nan)
figure, axis = plt.subplots(figsize=(ICLR_TEXT_WIDTH_IN, ICLR_TEXT_WIDTH_IN))
image = axis.imshow(lower_triangle, cmap=CMAP_CROSS, vmin=0.0, vmax=1.0)
for row in range(n_cross_signals):
    for column in range(row + 1):
        value = cross_r2[row, column]
        if np.isfinite(value):
            annotation = format_unit_decimal(value, 3)
            axis.text(
                column, row, annotation, ha="center", va="center",
                fontsize=3.0, color="white" if value > 0.62 else "#17171C",
            )
axis.set_xticks(range(n_cross_signals), cross_labels, rotation=90)
axis.set_yticks(range(n_cross_signals), cross_labels)
axis.tick_params(length=0, labelsize=4.7)
group_size = len(OPEN_MODELS)
for group_index, (family, _, _) in enumerate(CROSS_SIGNAL_SPECS):
    start = group_index * group_size
    center = start + (group_size - 1) / 2
    if group_index:
        boundary = start - 0.5
        axis.axhline(boundary, color="white", linewidth=0.8)
        axis.axvline(boundary, color="white", linewidth=0.8)
    axis.text(
        center, 1.09, family, transform=axis.get_xaxis_transform(),
        ha="center", va="bottom", fontsize=6.2, clip_on=False,
    )
    axis.text(
        -0.14, center, family, transform=axis.get_yaxis_transform(),
        ha="right", va="center", fontsize=6.2, clip_on=False,
    )
axis.set_xlim(-0.5, n_cross_signals - 0.5)
axis.set_ylim(n_cross_signals - 0.5, -0.5)
for spine in axis.spines.values():
    spine.set_visible(False)
colorbar = figure.colorbar(image, ax=axis, fraction=0.033, pad=0.02)
colorbar.set_label(r"Pearson $r^2$")
figure.subplots_adjust(left=0.22, right=0.88, bottom=0.14, top=0.88)
save_figure(figure, "fig6_all_measurements_heatmap.pdf")


### Appendix — Tuned-lens sensitivity

This companion analysis replaces the direct logit lens with a standard identity-initialized affine tuned lens trained by full-vocabulary $\mathrm{KL}(p_{\mathrm{final}}\,\|\,p_{\mathrm{lens}})$ on an official BoolQ training-split subsample; labels are never used. It evaluates the same 100 held-out validation prompts (569 user segments) for every model and uses the same signed, pooled-segment $F_{\mathrm{attr}}$, relative-depth interpolation, and paired prompt-cluster bootstrap as the primary plot. Solid curves are tuned lenses and dashed curves are the corresponding direct-logit-lens baseline on the identical observations.

This is an exploratory sensitivity analysis rather than a headline result: it uses seed 42 and model-specific training budgets (Qwen-0.5B: 512 prompts × 5 epochs/every layer; Qwen-3B and Llama-8B: 2,048 × 8/every second layer; Qwen-7B and Qwen-14B: 1,024 × 5/every fourth layer). The KL objective was still improving at the final epoch. The bootstrap therefore represents evaluation-sample uncertainty conditional on the fitted lenses, not lens-training uncertainty.


In [ ]:
tuned_depth = TUNED_LENS["relative_depth"].to_numpy(dtype=float)
assert len(TUNED_LENS) == 21 and TUNED_LENS["relative_depth"].is_unique
assert np.allclose(tuned_depth, np.linspace(0.0, 1.0, 21))
assert TUNED_LENS["n_models"].eq(5).all()
assert TUNED_LENS["n_model_pairs"].eq(10).all()
assert TUNED_LENS["n_prompts"].eq(100).all()
assert TUNED_LENS["n_user_segments"].eq(569).all()
assert TUNED_LENS["benchmark"].eq("boolq").all()
assert TUNED_LENS["pregrouper"].eq("sentence").all()
assert TUNED_LENS["statistic"].eq("pearson_r2").all()
assert TUNED_LENS["aggregation"].eq("mean_over_10_model_pairs_row_pooled").all()
assert TUNED_LENS["bootstrap_resamples"].eq(1000).all()
assert TUNED_LENS["confidence_level"].eq(0.95).all()
assert TUNED_LENS["seed"].eq(42).all()
assert TUNED_LENS["depth_alignment"].eq("linear_interpolation_block_outputs_only").all()
tuned_value_columns = [
    f"{lens}_{metric}{suffix}"
    for lens in ("untuned", "tuned")
    for metric in ("F_pred", "F_attr", "gap")
    for suffix in ("", "_ci_lower", "_ci_upper")
]
assert np.isfinite(TUNED_LENS[tuned_value_columns].to_numpy(dtype=float)).all()
for lens in ("untuned", "tuned"):
    assert np.allclose(
        TUNED_LENS[f"{lens}_gap"],
        TUNED_LENS[f"{lens}_F_pred"] - TUNED_LENS[f"{lens}_F_attr"],
    )
    for metric in ("F_pred", "F_attr", "gap"):
        assert (
            TUNED_LENS[f"{lens}_{metric}_ci_lower"]
            <= TUNED_LENS[f"{lens}_{metric}_ci_upper"]
        ).all()
endpoint = TUNED_LENS.iloc[-1]
for metric in ("F_pred", "F_attr", "gap"):
    for suffix in ("", "_ci_lower", "_ci_upper"):
        assert np.isclose(
            endpoint[f"tuned_{metric}{suffix}"],
            endpoint[f"untuned_{metric}{suffix}"],
        )

figure, axes = plt.subplots(1, 2, figsize=(ICLR_TEXT_WIDTH_IN, 2.45))
panel_fidelity, panel_gap = axes
for metric, color, label in (
    ("F_pred", PALETTE["F_pred"], r"$F_{\mathrm{pred}}$"),
    ("F_attr", PALETTE["F_attr"], r"$F_{\mathrm{attr}}$"),
):
    for lens, alpha, width in (("untuned", 0.06, 1.4), ("tuned", 0.16, 2.1)):
        prefix = f"{lens}_{metric}"
        panel_fidelity.fill_between(
            tuned_depth,
            TUNED_LENS[f"{prefix}_ci_lower"],
            TUNED_LENS[f"{prefix}_ci_upper"],
            color=color,
            alpha=alpha,
            linewidth=0.0,
            zorder=1,
        )
        panel_fidelity.plot(
            tuned_depth,
            TUNED_LENS[prefix],
            color=color,
            linestyle=LENS_LINESTYLES[lens],
            linewidth=width,
            alpha=1.0 if lens == "tuned" else 0.72,
            label=f"{lens.title()} {label}",
            zorder=3,
        )
panel_fidelity.set_ylim(0.0, 1.0)
panel_fidelity.set_ylabel(r"Mean pairwise Pearson $r^2$")

tuned_gap = TUNED_LENS["tuned_gap"].to_numpy(dtype=float)
panel_gap.fill_between(
    tuned_depth,
    TUNED_LENS["untuned_gap_ci_lower"],
    TUNED_LENS["untuned_gap_ci_upper"],
    color=PALETTE["control"],
    alpha=0.06,
    linewidth=0.0,
    zorder=1,
)
panel_gap.fill_between(
    tuned_depth,
    TUNED_LENS["tuned_gap_ci_lower"],
    TUNED_LENS["tuned_gap_ci_upper"],
    color=PALETTE["control"],
    alpha=0.18,
    linewidth=0.0,
    label="Tuned: 95% paired prompt-bootstrap CI",
    zorder=1,
)
panel_gap.fill_between(
    tuned_depth, 0.0, tuned_gap, where=tuned_gap >= 0.0,
    color=PALETTE["F_pred"], alpha=0.18, linewidth=0.0, zorder=2,
)
panel_gap.fill_between(
    tuned_depth, 0.0, tuned_gap, where=tuned_gap < 0.0,
    color=PALETTE["F_attr"], alpha=0.18, linewidth=0.0, zorder=2,
)
panel_gap.plot(
    tuned_depth, tuned_gap, color=PALETTE["neutral"], linewidth=2.1,
    label="Tuned gap", zorder=3,
)
panel_gap.plot(
    tuned_depth, TUNED_LENS["untuned_gap"], color=PALETTE["control"],
    linestyle=LENS_LINESTYLES["untuned"], linewidth=1.5,
    label="Untuned gap", zorder=3,
)
panel_gap.axhline(0.0, color=PALETTE["control"], linewidth=0.9, zorder=0)
gap_bounds = TUNED_LENS[["tuned_gap_ci_lower", "tuned_gap_ci_upper", "untuned_gap_ci_lower", "untuned_gap_ci_upper"]].to_numpy(dtype=float)
gap_limit = min(1.0, np.ceil(max(0.2, np.abs(gap_bounds).max()) * 20.0) / 20.0)
panel_gap.set_ylim(-gap_limit, gap_limit)
panel_gap.set_ylabel(r"Mean pairwise $r^2$ gap")

for axis in axes:
    axis.set_xlim(0.0, 1.0)
    axis.set_xlabel("Relative decoder depth")
    axis.legend(loc="best", frameon=False)
figure.tight_layout()
save_figure(figure, "appendix_tuned_lens_sensitivity.pdf")

display(
    TUNED_LENS.loc[
        TUNED_LENS["relative_depth"].isin([0.6, 0.7, 0.8, 0.9, 1.0]),
        ["relative_depth", "untuned_F_pred", "untuned_F_attr", "tuned_F_pred", "tuned_F_attr", "tuned_gap"],
    ].round(3)
)


## Cross-benchmark summary

Binary tasks report median pairwise Pearson $r^2$. ANLI prediction and attribution use centered multivariate RV over its three pairwise label margins; its scalar representation and cross-fidelity metrics retain the entailment-minus-contradiction readout. RACE prediction and attribution use centered multivariate RV over all six A–D margins; its scalar representation and cross-fidelity metrics use centered RV on the answer-conditioned correct-vs-rest contrast. LAMBADA is omitted because three hosted APIs do not reliably expose the teacher-forced target-token log-probabilities required by that benchmark.


In [ ]:
BENCHMARK_COLUMNS = [
    ("BoolQ", "boolq", "sentence"),
    ("ANLI R1", "anli_r1", "sentence"),
    ("ANLI R2", "anli_r2", "sentence"),
    ("ANLI R3", "anli_r3", "sentence"),
    ("WinoGrande", "winogrande", "sentence"),
    ("BoolQ word", "boolq", "word"),
]
TABLE_METRICS = [metric for _, metrics in SUMMARY_GROUPS for metric in metrics]


def _benchmark_median(metric: str, benchmark: str, pregrouper: str) -> tuple[float | None, int]:
    if benchmark.startswith("anli_") and metric in {"F_pred", "F_attr"}:
        values = pd.to_numeric(
            ANLI_RV.loc[
                ANLI_RV["benchmark"].eq(benchmark)
                & ANLI_RV["scope"].eq("all")
                & ANLI_RV["metric"].eq(f"{metric}_rv"),
                "f_point",
            ], errors="coerce"
        ).dropna()
        return (float(values.median()), len(values)) if len(values) else (None, 0)
    rows = canonical_rows(
        benchmark=benchmark, pregrouper=pregrouper, metric=metric
    )
    if metric in {"F_pred", "F_attr"}:
        populations = {"open_open", "open_hosted", "hosted_hosted"}
    elif metric.endswith("_to_attr"):
        populations = {"open_open", "open_hosted"}
    else:
        populations = {"open_open"}
    values = pd.to_numeric(
        rows.loc[rows["pair_population"].isin(populations), "f_point"], errors="coerce"
    ).dropna()
    return (float(values.median()), len(values)) if len(values) else (None, 0)


cross_records = []
count_records = []
for metric in TABLE_METRICS:
    record = {"Metric": METRIC_LABELS[metric]}
    counts = {"Metric": METRIC_LABELS[metric]}
    for label, benchmark, pregrouper in BENCHMARK_COLUMNS:
        value, count = _benchmark_median(metric, benchmark, pregrouper)
        record[label] = "---" if value is None else format_unit_decimal(value, 3)
        counts[label] = count
    race_metric = {
        "F_pred": "F_pred_rv",
        "F_attr": "F_attr_rv",
        "F_attn_mean": "F_attn_mean_rv",
        "F_attn_max": "F_attn_max_rv",
        "F_attn_rollout": "F_attn_rollout_rv",
        "F_mag": "F_mag_rv",
        "F_align": "F_align_rv",
        "F_mag_to_attr": "F_mag_to_attr_rv",
        "F_align_to_attr": "F_align_to_attr_rv",
        "F_attn_mean_to_attr": "F_attn_mean_to_attr_rv",
        "F_attn_max_to_attr": "F_attn_max_to_attr_rv",
        "F_attn_rollout_to_attr": "F_attn_rollout_to_attr_rv",
    }[metric]
    race_representation = (
        "all_pairs" if metric in {"F_pred", "F_attr"} else "canonical_scalar"
    )
    race_values = pd.to_numeric(
        RACE_RV.loc[
            RACE_RV["representation"].eq(race_representation)
            & RACE_RV["metric"].eq(race_metric),
            "f_point",
        ], errors="coerce"
    ).dropna()
    record["RACE RV"] = format_unit_decimal(float(race_values.median()), 3)
    counts["RACE RV"] = len(race_values)
    cross_records.append(record)
    count_records.append(counts)

CROSS_BENCHMARK_TABLE = pd.DataFrame(cross_records)
CROSS_BENCHMARK_COUNTS = pd.DataFrame(count_records)

# Pair-count assertions keep partial public coverage visible instead of silently
# changing the population summarized by a table cell.
for record in count_records:
    metric_label = record["Metric"]
    metric = next(key for key, label in METRIC_LABELS.items() if label == metric_label)
    for label, _, _ in BENCHMARK_COLUMNS:
        if metric in {"F_pred", "F_attr"}:
            expected = 55
        elif metric.endswith("_to_attr"):
            expected = 50
        else:
            expected = 10
        assert record[label] == expected, (metric, label, record[label], expected)
    race_expected = (
        55 if metric in {"F_pred", "F_attr"}
        else 50 if metric.endswith("_to_attr")
        else 10
    )
    assert record["RACE RV"] == race_expected, (metric, record["RACE RV"], race_expected)

race_all_pairs = RACE_RV.loc[RACE_RV["representation"].eq("all_pairs")]
race_coverage_min = float(race_all_pairs["observation_coverage"].min())
race_coverage_max = float(race_all_pairs["observation_coverage"].max())
assert np.isclose(race_coverage_min, 0.9798548892431156)
assert np.isclose(race_coverage_max, 1.0)

display(CROSS_BENCHMARK_TABLE)
print("Pair counts (availability is pair-specific):")
display(CROSS_BENCHMARK_COUNTS)

columns = [label for label, _, _ in BENCHMARK_COLUMNS] + ["RACE RV"]
lines = [
    r"% Full-dialog canonical medians. BoolQ/WinoGrande: Pearson r^2; ANLI/RACE: centered RV.",
    r"\begin{tabular}{l" + "c" * len(columns) + "}",
    r"\toprule",
    "Metric & " + " & ".join(columns) + r" \\",
    r"\midrule",
]
cross_group_breaks = set(np.cumsum([len(metrics) for _, metrics in SUMMARY_GROUPS])[:-1])
for record_idx, record in enumerate(cross_records):
    if record_idx in cross_group_breaks:
        lines.append(r"\midrule")
    lines.append(record["Metric"] + " & " + " & ".join(record[column] for column in columns) + r" \\")
lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
])
(TABLES_DIR / "cross_benchmark_pearson_cameraready.tex").write_text("\n".join(lines) + "\n")
print(TABLES_DIR / "cross_benchmark_pearson_cameraready.tex")


## Figure 23 — Public RACE fidelity heatmap

In [ ]:
def race_grid(metric: str) -> np.ndarray:
    frame = RACE_RV.loc[
        RACE_RV["representation"].eq("all_pairs") & RACE_RV["metric"].eq(metric)
    ]
    grid = np.full((len(MODELS), len(MODELS)), np.nan)
    model_index = {model: index for index, model in enumerate(MODELS)}
    for row in frame.itertuples(index=False):
        first, second = model_index[row.model_s], model_index[row.model_t]
        grid[first, second] = grid[second, first] = float(row.f_point)
    return grid


race_pred = race_grid("F_pred_rv")
race_attr = race_grid("F_attr_rv")
indices = np.indices(race_pred.shape)
fig, ax = plt.subplots(figsize=(4.15, 3.95))
ax.imshow(np.where(indices[0] < indices[1], race_pred, np.nan), cmap=CMAP_PRED, vmin=0.0, vmax=1.0)
ax.imshow(np.where(indices[0] > indices[1], race_attr, np.nan), cmap=CMAP_ATTR, vmin=0.0, vmax=1.0)
for row in range(len(MODELS)):
    for column in range(len(MODELS)):
        if row == column:
            continue
        value = race_pred[row, column] if row < column else race_attr[row, column]
        if np.isfinite(value):
            ax.text(column, row, format_unit_decimal(value, 2), ha="center", va="center", fontsize=5.8,
                    color="white" if value > 0.68 else "#17171C")
ax.set_xticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS], rotation=48, ha="right")
ax.set_yticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
save_figure(fig, "fig23_race_rv_heatmap.pdf")


## Appendix: multiclass missing-log-probability sensitivity

For ANLI and RACE, this full-dialog sensitivity assigns partially censored hosted top-$k$ label scores a shared model-and-benchmark floor while leaving all-label-missing rows unavailable. The horizontal axis moves that floor below the lowest observed finite label score. Prediction and attribution fidelity move together across the sweep, preserving the coarse ordering. Dashed horizontal lines show the canonical model-specific finite-extreme estimates; gray vertical bands mark the range of selected offsets, with their medians dotted. Representation quantities are unaffected by this postprocessing and therefore provide horizontal references. Colored bands show the interquartile range across model pairs.


In [ ]:
from benchmark_scripts.plot_missingness_sensitivity import plot_sensitivity, reference_values

plot_sensitivity(
    MULTICLASS_FLOOR,
    reference_values(str(RESULTS_DIR)),
    str(FIGURES_DIR / "appendix_multiclass_floor_sensitivity.pdf"),
)


## Output inventory

In [ ]:
EXPECTED_OUTPUTS = [
    TABLES_DIR / "summary_pearson.tex",
    TABLES_DIR / "cross_benchmark_pearson_cameraready.tex",
    FIGURES_DIR / "fig3a_fidelity_heatmap.pdf",
    FIGURES_DIR / "fig3b_logodds_contour.pdf",
    FIGURES_DIR / "fig3c_attribution_contour.pdf",
    FIGURES_DIR / "fig4a_delta_norm_contour.pdf",
    FIGURES_DIR / "fig4b_alignment_contour.pdf",
    FIGURES_DIR / "fig4c_norm_contribution_contour.pdf",
    FIGURES_DIR / "fig5_per_layer_fidelity.pdf",
    FIGURES_DIR / "fig6_all_measurements_heatmap.pdf",
    FIGURES_DIR / "fig23_race_rv_heatmap.pdf",
    FIGURES_DIR / "appendix_tuned_lens_sensitivity.pdf",
    FIGURES_DIR / "appendix_multiclass_floor_sensitivity.pdf",
]
missing = [path for path in EXPECTED_OUTPUTS if not path.is_file() or path.stat().st_size == 0]
assert not missing, missing
print("Generated", len(EXPECTED_OUTPUTS), "paper outputs")
for path in EXPECTED_OUTPUTS:
    print(f"{path.relative_to(OUTPUT_ROOT)}\t{path.stat().st_size:,} bytes")
